In [1]:
from pathlib import Path
from shutil import copy2
from typing import Callable, Mapping, Optional, Union

import hydra

from malpolon.data.data_module import RLSDataModule, RLSDataset
from malpolon.logging import Summary
from malpolon.models.custom_models import MultiModalModel
from malpolon.models.standard_prediction_systems import GenericPredictionSystem
from captum.attr import IntegratedGradients, Saliency

from omegaconf import DictConfig, OmegaConf

import lightning.pytorch as pl
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint

import torch
from torch import Tensor, nn
from torch.utils.data import DataLoader

import torchmetrics.functional as Fmetrics
import os
import numpy as np
import pandas as pd
import copy

/home/gmorand/venvs/deepsdm2/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OmegaConf.register_new_resolver("eval", eval)

def get_custom_metric(nbins, average_type):

    def custom_metric(predictions, target):

        predictions = predictions.argmax(dim=-1)
        return Fmetrics.classification.multiclass_accuracy(predictions, target, num_classes=nbins, average=average_type)

    return custom_metric
    
class PresenceSystem(GenericPredictionSystem):
    def __init__(
        self,
        submodels: DictConfig,
        num_species: int,
        num_bins: int,
        freeze_submodels: bool,
        loss: Union[torch.nn.modules.loss._Loss, str] = "ce_and_sr_loss",
        optimizer: Union[torch.nn.Module, Mapping] = None,
        metrics: Optional[dict[str, Callable]] = None,
        loss_kwargs: Optional[Mapping] = {},
        alpha: Optional[float] = None
    ):

        model = MultiModalModel(
            submodels,
            num_species,
            num_bins,
            freeze_submodels
        )

        metrics = {'micro_acc': get_custom_metric(num_bins, 'micro'),
                   'macro_acc': get_custom_metric(num_bins, 'macro')}

        if alpha is not None:
            loss_kwargs['alpha'] = alpha

        super().__init__(model, loss, optimizer, loss_kwargs, metrics=metrics)

        self.model = model


    def remove_final_layer(self):
        """Remove the final layers of the model to keep only the feature extractor."""

        self.model.aggregator_model[1] = nn.Identity()

In [3]:
from hydra import compose, initialize
from omegaconf import OmegaConf

CKPT_NAME = '18_humenv_dhw_cat_pa-2025-07-15_13-37-preds'

with initialize(version_base=None, config_path="../../../../../marbec-data/RLS-Australia/malpolon/logs/" + CKPT_NAME, job_name="hparams"):
    cfg = compose(config_name="hparams")

#print(cfg)

datamodule = RLSDataModule(**cfg.data, modality_names= list(cfg.model.submodels.keys()),
                               target_transform=lambda x: (x != 0).astype(float))

loss_kwargs = {'num_bins': cfg.model.num_bins,
                   'num_species': cfg.model.num_species,
                   'loss_weights': None}#datamodule.get_class_weights()}

reg_system = PresenceSystem(**cfg.model, **cfg.optim, loss_kwargs=loss_kwargs)

trainer = pl.Trainer(**cfg.trainer)

model_loaded = PresenceSystem.load_from_checkpoint(cfg.run.checkpoint_path)

Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [4]:
ds = RLSDataset(
            datamodule.root,
            datamodule.dataset_name,
            datamodule.inputs_path,
            "train+val",
            datamodule.num_classes,
            patch_data=datamodule.modality_names,
            transform=datamodule.test_transform,
            target_transform=datamodule.target_transform
        )

dl = DataLoader(ds,
            sampler=datamodule.sampler,
            batch_size=datamodule.inference_batch_size,
            num_workers=datamodule.num_workers,
            pin_memory=datamodule.pin_memory,
        )

predictions = trainer.predict(dataloaders=dl, model=model_loaded)
predictions = torch.cat(predictions)

/home/gmorand/venvs/deepsdm2/lib/python3.12/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/home/gmorand/venvs/deepsdm2/lib/python3.12/site-packages/lightning/pytorch/utilities/parsing.py:45: Attribute 'metrics' removed from hparams because it cannot be pickled. You can suppress this warning by setting `self.save_hyperparameters(ignore=['metrics'])`.
You are using a CUDA device ('NVIDIA A40') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable

Predicting DataLoader 0: 100%|██████████| 1406/1406 [02:38<00:00,  8.89it/s]


In [5]:
predictions.shape

torch.Size([22484, 1796, 2])

In [6]:
out_dir="../../../../../marbec-data/RLS-Australia/malpolon/outputs/" + CKPT_NAME
out_name="predictions_trainval"

predictions = predictions.softmax(dim=-1)[..., 1]
df = pd.DataFrame(index=ds.survey_ids,
                  columns=ds.species,
                  data=predictions)

df.to_csv(Path(out_dir) / Path(out_name + ".csv"), sep=',')

In [7]:
df.describe()

,Abalistes stellatus,Abudefduf bengalensis,Abudefduf septemfasciatus,Abudefduf sexfasciatus,Abudefduf sordidus,Abudefduf vaigiensis,Abudefduf whitleyi,Acanthaluteres brownii,Acanthaluteres spilomelanurus,Acanthaluteres vittiger,...,Vincentia punctata,Xiphasia setifer,Zabidius novemaculeatus,Zanclus cornutus,Zebrasoma desjardinii,Zebrasoma scopas,Zebrasoma velifer,Zoramia leptacanthus,Zoramia spp.,Zoramia viridiventer
count,2.248400e+04,22484.000000,2.248400e+04,22484.000000,2.248400e+04,22484.000000,2.248400e+04,2.248400e+04,2.248400e+04,22484.000000,...,2.248400e+04,2.248400e+04,2.248400e+04,22484.000000,2.248400e+04,22484.000000,22484.000000,2.248400e+04,2.248400e+04,22484.000000
mean,4.056068e-04,0.027425,3.313120e-06,0.029969,4.807470e-04,0.023391,6.386636e-03,1.976309e-02,9.182654e-03,0.168807,...,9.652035e-05,3.123505e-06,1.837412e-04,0.081093,6.139804e-05,0.070101,0.047428,3.094696e-04,3.956213e-06,0.001044
std,3.139014e-03,0.115967,7.421536e-06,0.088327,2.738723e-03,0.051089,3.691161e-02,6.691726e-02,2.238205e-02,0.224517,...,8.994158e-04,6.879009e-06,2.982180e-03,0.175867,1.468314e-03,0.181906,0.127554,2.678355e-03,8.961312e-06,0.007039
min,1.211237e-08,0.000001,7.190940e-08,0.000007,2.283189e-07,0.000022,3.743934e-07,2.579760e-08,5.338515e-07,0.000003,...,2.198222e-08,5.358540e-08,1.760123e-08,0.000016,3.433438e-08,0.000004,0.000007,1.515450e-07,8.314140e-08,0.000001
25%,9.531751e-07,0.000056,7.036121e-07,0.000149,3.386770e-06,0.000173,2.227437e-05,2.092202e-05,3.552139e-04,0.012562,...,1.108886e-06,6.379830e-07,9.551720e-07,0.000326,1.279695e-06,0.000136,0.000119,2.632320e-06,7.644501e-07,0.000013
50%,4.978472e-06,0.000130,1.314713e-06,0.000449,8.714613e-06,0.000581,9.051538e-05,2.172327e-04,1.371392e-03,0.057356,...,4.108487e-06,1.177763e-06,3.763601e-06,0.001122,3.074548e-06,0.000296,0.000236,7.353412e-06,1.456476e-06,0.000030
75%,2.924377e-05,0.001482,2.726318e-06,0.007862,4.360496e-05,0.024772,4.195135e-04,9.315916e-04,7.449667e-03,0.241410,...,2.462788e-05,2.558587e-06,1.639046e-05,0.035335,8.581915e-06,0.002689,0.001117,2.837111e-05,3.122096e-06,0.000089
max,6.970712e-02,0.931548,3.233341e-04,0.766128,5.501861e-02,0.590604,6.022192e-01,6.547291e-01,3.450747e-01,0.873788,...,3.588992e-02,2.694016e-04,2.800830e-01,0.839439,7.855996e-02,0.890655,0.758917,1.266539e-01,2.404853e-04,0.232863
